In [1]:
# ============================================================
# Sincronización de luciérnagas — Red discreta de Kuramoto
# ============================================================
#
# Modelo:
#   θᵢᵗ⁺¹ = (θᵢᵗ + ω + ⌊K · Σⱼ∈N(i) δM(θⱼ,θᵢ)⌋) mod M
#
# Topologías:
#   Camino · Ciclo · Completo · Estrella · Aleatorio
#   Cuadrícula (Von Neumann) · Hexagonal
#
# En esta versión:
# - Se implementa de verdad G_θ^ε usando distancia circular.
# - Se distingue entre sincronización exacta y ε-sincronización.
# - La red aleatoria se fuerza a ser conexa.
# - K, ω y ε se comportan como controles "hot".
# ============================================================

import math
import warnings
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgs

from matplotlib.patches import Circle
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")


# ============================================================
# 0. PALETA
# ============================================================
CMAP_PHASE  = plt.cm.hsv
C_EDGE_BASE = "#cccccc"
C_EDGE_EPS  = "#185FA5"
C_NODE_EPS  = "#185FA5"
C_NODE_ISO  = "#C62828"
C_R         = "#185FA5"
C_A         = "#BA7517"
C_OK        = "#2E7D32"
C_WARN      = "#E65100"
C_BAD       = "#B71C1C"
C_SYNC_BG   = "#F1F8E9"


# ============================================================
# 1. ARITMÉTICA EN Z_M
# ============================================================
def d_signed(a: int, b: int, M: int) -> int:
    """
    Diferencia circular con signo δ_M(a,b) en un representante centrado.
    Intenta devolver un valor en torno a (-M/2, M/2].
    """
    a = int(a)
    b = int(b)
    return ((a - b + M // 2) % M) - (M // 2)


def d_circ(a: int, b: int, M: int) -> int:
    """
    Distancia circular sin signo en Z_M.
    """
    a = int(a)
    b = int(b)
    r = abs(a - b) % M
    return min(r, M - r)


# ============================================================
# 2. CONSTRUCCIÓN DEL GRAFO
# ============================================================
def grid_dims(n: int):
    """
    Dimensiones (filas, cols) aproximadamente cuadradas para n nodos.
    """
    cols = max(1, round(math.sqrt(n)))
    rows = math.ceil(n / cols)
    while cols * rows < n:
        rows += 1
    return rows, cols


def _force_connectivity_random_graph(G: nx.Graph, seed: int = 7):
    """
    Une componentes aleatorias hasta que el grafo quede conexo.
    """
    rng = np.random.default_rng(seed)
    comps = list(nx.connected_components(G))
    while len(comps) > 1:
        c1 = list(comps[0])
        c2 = list(comps[1])
        u = int(rng.choice(c1))
        v = int(rng.choice(c2))
        G.add_edge(u, v)
        comps = list(nx.connected_components(G))


def build_graph(topo: str, N: int, p: float = 0.35, seed: int = 7) -> nx.Graph:
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    G.add_nodes_from(range(N))
    G.graph["topo"] = topo

    if topo == "camino":
        for i in range(N - 1):
            G.add_edge(i, i + 1)

    elif topo == "ciclo":
        for i in range(N):
            G.add_edge(i, (i + 1) % N)

    elif topo == "completo":
        for i in range(N):
            for j in range(i + 1, N):
                G.add_edge(i, j)

    elif topo == "estrella":
        for i in range(1, N):
            G.add_edge(0, i)

    elif topo == "aleatorio":
        for i in range(N):
            for j in range(i + 1, N):
                if rng.random() < p:
                    G.add_edge(i, j)
        _force_connectivity_random_graph(G, seed=seed)

    elif topo == "grid":
        rows, cols = grid_dims(N)
        for r in range(rows):
            for c in range(cols):
                i = r * cols + c
                if i >= N:
                    continue
                if c + 1 < cols and i + 1 < N:
                    G.add_edge(i, i + 1)
                if r + 1 < rows and i + cols < N:
                    G.add_edge(i, i + cols)

    elif topo == "hex":
        rows, cols = grid_dims(N)
        deltas_even = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, -1), (-1, -1)]
        deltas_odd  = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (-1, 1)]

        for r in range(rows):
            for c in range(cols):
                i = r * cols + c
                if i >= N:
                    continue
                deltas = deltas_odd if r % 2 == 1 else deltas_even
                for dr, dc in deltas:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols:
                        j = nr * cols + nc
                        if j < N:
                            G.add_edge(i, j)
    else:
        raise ValueError(f"Topología desconocida: {topo}")

    return G


# ============================================================
# 3. POSICIONES DE NODOS
# ============================================================
def build_positions(topo: str, N: int) -> dict:
    pad = 0.12
    pos = {}

    if topo in ("grid", "hex"):
        rows, cols = grid_dims(N)
        cw = (1.0 - 2 * pad) / max(cols - 1, 1)
        ch = (1.0 - 2 * pad) / max(rows - 1, 1)

        for r in range(rows):
            for c in range(cols):
                i = r * cols + c
                if i >= N:
                    continue
                hx = 0.5 * cw if (topo == "hex" and r % 2 == 1) else 0.0
                pos[i] = (pad + c * cw + hx, 1.0 - pad - r * ch)

    elif topo == "estrella":
        pos[0] = (0.5, 0.5)
        rad = 0.5 - pad
        for i in range(1, N):
            a = ((i - 1) / max(N - 1, 1)) * 2 * math.pi
            pos[i] = (0.5 + rad * math.cos(a), 0.5 + rad * math.sin(a))

    elif topo == "camino":
        cols_per_row = min(max(5, math.ceil(math.sqrt(N))), N)
        cw = (1.0 - 2 * pad) / max(cols_per_row - 1, 1)
        n_rows = math.ceil(N / cols_per_row)
        row_h = (1.0 - 2 * pad) / max(n_rows - 1, 1)

        for i in range(N):
            row, col = divmod(i, cols_per_row)
            pos[i] = (pad + col * cw, 1.0 - pad - row * row_h)

    else:
        rad = 0.5 - pad
        for i in range(N):
            a = (i / max(N, 1)) * 2 * math.pi - math.pi / 2
            pos[i] = (0.5 + rad * math.cos(a), 0.5 + rad * math.sin(a))

    return pos


# ============================================================
# 4. CONDICIÓN INICIAL
# ============================================================
def make_theta0(N: int, M: int, topo: str, mode: str = "bloques", seed: int = 7) -> np.ndarray:
    rng = np.random.default_rng(seed)

    if mode == "aleatoria":
        return rng.integers(0, M, N, dtype=int)

    if mode == "casi_sincronizada":
        th = np.zeros(N, dtype=int)
        if N > 1:
            th[-1] = 1 % M
        return th

    if mode == "gradiente":
        return (np.arange(N, dtype=int) % M)

    # modo por defecto: bloques
    th = np.zeros(N, dtype=int)
    if topo in ("grid", "hex"):
        rows, cols = grid_dims(N)
        for r in range(rows):
            for c in range(cols):
                i = r * cols + c
                if i >= N:
                    continue
                th[i] = 0 if r < rows // 2 else M // 2
    else:
        th[N // 2:] = M // 2

    return th % M


# ============================================================
# 5. DINÁMICA DE KURAMOTO DISCRETA
# ============================================================
def kuramoto_step(G: nx.Graph, theta: np.ndarray, M: int, K: float, omega: int) -> np.ndarray:
    new = theta.copy()
    for i in G.nodes():
        coup = sum(d_signed(theta[j], theta[i], M) for j in G.neighbors(i))
        new[i] = (int(theta[i]) + int(omega) + int(math.floor(K * coup))) % M
    return new


def simulate(G: nx.Graph, theta0: np.ndarray, M: int, K: float, omega: int, T: int) -> np.ndarray:
    history = [theta0.copy()]
    th = theta0.copy()
    for _ in range(T):
        th = kuramoto_step(G, th, M, K, omega)
        history.append(th.copy())
    return np.array(history, dtype=int)


# ============================================================
# 6. MÉTRICAS
# ============================================================
def order_parameter(theta: np.ndarray, M: int) -> float:
    z = np.exp(2j * np.pi * theta / M)
    return float(np.abs(z.mean()))


def edge_agreement(G: nx.Graph, theta: np.ndarray, M: int, eps: int):
    """
    Cuenta aristas {u,v} tales que d_circ(theta_u, theta_v) <= eps.
    """
    cnt = sum(1 for u, v in G.edges() if d_circ(theta[u], theta[v], M) <= eps)
    return cnt, G.number_of_edges()


def is_exact_sync(theta: np.ndarray) -> bool:
    return bool(np.all(theta == theta[0]))


def is_eps_sync(theta: np.ndarray, M: int, eps: int) -> bool:
    n = len(theta)
    for i in range(n):
        for j in range(i + 1, n):
            if d_circ(theta[i], theta[j], M) > eps:
                return False
    return True


def first_sync_time(history: np.ndarray, M: int, eps: int, exact: bool = False):
    for t, th in enumerate(history):
        ok = is_exact_sync(th) if exact else is_eps_sync(th, M, eps)
        if ok:
            return t
    return None


def eps_subgraph(G: nx.Graph, theta: np.ndarray, M: int, eps: int) -> nx.Graph:
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    for u, v in G.edges():
        if d_circ(theta[u], theta[v], M) <= eps:
            H.add_edge(u, v)
    return H


# ============================================================
# 7. DIBUJO DE LA RED
# ============================================================
def _node_r(topo: str, N: int) -> float:
    if topo in ("grid", "hex"):
        rows, cols = grid_dims(N)
        cw = 0.76 / max(cols - 1, 1)
        ch = 0.76 / max(rows - 1, 1)
        return max(0.022, min(0.055, min(cw, ch) * 0.36))
    return 0.055 if N <= 10 else 0.040


# ------------------------------------------------------------
# INTERPRETACIÓN DEL COLOR DE LOS NODOS
# ------------------------------------------------------------
# Cada nodo i tiene un estado θ_i ∈ {0,1,...,M-1} que representa su fase.
#
# La visualización utiliza un mapa de colores (CMAP_PHASE = hsv) para codificar
# estas fases de manera circular:
#
#   θ_i = 0   → amarillo brillante (evento de "destello")
#   θ_i ≠ 0   → color asignado por CMAP_PHASE(θ_i / (M-1))
#
# Por tanto:
#   - Cambiar de color = cambio de fase del oscilador.
#   - Colores iguales entre nodos = mismas fases.
#   - Amarillo indica sincronización instantánea en el estado de disparo.
#
# Esta representación permite visualizar la convergencia hacia sincronización
# como la homogeneización de colores en la red.
# ------------------------------------------------------------



def draw_network(ax: plt.Axes, G: nx.Graph, H: nx.Graph,
                 pos: dict, theta: np.ndarray, M: int,
                 show_eps: bool = True, title: str = ""):
    ax.clear()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=9, pad=4)

    topo = G.graph.get("topo", "")
    R = _node_r(topo, G.number_of_nodes())
    vmax = max(1, M - 1)

    # Aristas
    for u, v in G.edges():
        xu, yu = pos[u]
        xv, yv = pos[v]
        if show_eps and H.has_edge(u, v):
            ax.plot([xu, xv], [yu, yv], color=C_EDGE_EPS,
                    lw=2.0, zorder=1, solid_capstyle="round")
        else:
            ax.plot([xu, xv], [yu, yv], color=C_EDGE_BASE,
                    lw=0.8, zorder=1, alpha=0.7)

    # Nodos
    for i in G.nodes():
        x, y = pos[i]
        ph = int(theta[i])
        flashing = (ph == 0)
        col = "#FFD600" if flashing else CMAP_PHASE(ph / vmax)

        if flashing:
            ax.add_patch(Circle((x, y), R * 1.6, color="#FFFDE7", zorder=2, alpha=0.5))

        ax.add_patch(Circle((x, y), R, color=col, ec="#444", linewidth=0.6, zorder=3))

        if show_eps:
            degH = H.degree(i)
            ec = C_NODE_EPS if degH > 0 else C_NODE_ISO
            ls = "-" if degH > 0 else "--"
            ax.add_patch(Circle((x, y), R * 1.22, fill=False,
                                ec=ec, linewidth=1.8, linestyle=ls, zorder=4))

        if R >= 0.030:
            fc = "#1A237E" if flashing else "white"
            ax.text(x, y, str(ph), ha="center", va="center",
                    fontsize=max(5, int(R * 120)),
                    color=fc, fontweight="bold", zorder=5)


def draw_metrics(ax: plt.Axes, r_hist: list, a_hist: list, t_current: int, sync_t):
    ax.clear()
    ts = range(len(r_hist))

    ax.plot(ts, r_hist, lw=1.6, marker="o", ms=2.5, label="r(t) — orden")
    ax.plot(ts, a_hist, lw=1.6, marker="s", ms=2.5, linestyle="--", label="acuerdo ε(t)")

    ax.axvline(t_current, color="#555", lw=1.0, ls=":", zorder=3)

    if sync_t is not None:
        ax.axvline(sync_t, color=C_OK, lw=1.4, alpha=0.8, label=f"ε-sync t={sync_t}")
        ax.axvspan(sync_t, max(len(r_hist) - 1, sync_t + 1), color=C_OK, alpha=0.07)
        ax.text(sync_t + 0.2, 0.05, f"t*={sync_t}", fontsize=7, color=C_OK, va="bottom")

    ax.set_ylim(-0.02, 1.10)
    ax.set_xlabel("t", fontsize=9)
    ax.legend(fontsize=7.5, loc="upper right")
    ax.grid(True, alpha=0.2)
    ax.tick_params(labelsize=8)


def draw_subgraph_row(fig: plt.Figure, gs_cell,
                      G: nx.Graph, pos: dict, history: np.ndarray,
                      M: int, eps: int, sample_times: list, t_current: int):
    n = len(sample_times)
    subgs = gs_cell.subgridspec(1, n, wspace=0.05)
    topo = G.graph.get("topo", "")
    vmax = max(1, M - 1)

    for idx, t in enumerate(sample_times):
        ax = fig.add_subplot(subgs[0, idx])
        theta = history[t]
        H = eps_subgraph(G, theta, M, eps)
        synced = is_eps_sync(theta, M, eps)
        R = _node_r(topo, G.number_of_nodes())

        if synced:
            ax.set_facecolor(C_SYNC_BG)

        for u, v in G.edges():
            xu, yu = pos[u]
            xv, yv = pos[v]
            if H.has_edge(u, v):
                ax.plot([xu, xv], [yu, yv], color=C_EDGE_EPS, lw=1.4, zorder=1)
            else:
                ax.plot([xu, xv], [yu, yv], color=C_EDGE_BASE, lw=0.5, alpha=0.6, zorder=1)

        for i in G.nodes():
            x, y = pos[i]
            ph = int(theta[i])
            col = "#FFD600" if ph == 0 else CMAP_PHASE(ph / vmax)
            ax.add_patch(Circle((x, y), R * 0.82, color=col, ec="#555", linewidth=0.3, zorder=2))

        lbl = f"t={t}" + (" ✓" if synced else "")
        ax.set_title(lbl, fontsize=6.5, pad=1.5, color=C_OK if synced else "black")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal")
        ax.axis("off")

        if t == t_current:
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor("#333")
                spine.set_linewidth(1.6)


# ============================================================
# 8. FIGURA PRINCIPAL
# ============================================================
def make_figure(G: nx.Graph, history: np.ndarray,
                t: int, M: int, eps: int, pos: dict,
                r_hist: list, a_hist: list, sync_t):
    theta = history[t]
    H = eps_subgraph(G, theta, M, eps)

    eps_synced = is_eps_sync(theta, M, eps)
    exact_synced = is_exact_sync(theta)

    cnt, tot = edge_agreement(G, theta, M, eps)
    T = len(history) - 1

    n_samp = min(8, len(history))
    stimes = list(np.unique(np.linspace(0, T, n_samp, dtype=int)))
    if sync_t is not None and sync_t not in stimes:
        stimes.append(sync_t)
    if t not in stimes:
        stimes.append(t)
    stimes = sorted(set(stimes))[:8]

    fig = plt.figure(figsize=(17, 9))
    gs = mgs.GridSpec(
        2, 3, figure=fig,
        height_ratios=[1.1, 0.9],
        width_ratios=[1, 1, 1.1],
        hspace=0.42, wspace=0.28
    )

    ax_net = fig.add_subplot(gs[0, 0])
    ax_eps = fig.add_subplot(gs[0, 1])
    ax_met = fig.add_subplot(gs[0, 2])

    sync_str = "  [ε-sync ✓]" if eps_synced else ""
    draw_network(ax_net, G, H, pos, theta, M, show_eps=False, title=f"Red,  t = {t}{sync_str}")
    if eps_synced:
        ax_net.set_facecolor(C_SYNC_BG)

    draw_network(
        ax_eps, G, H, pos, theta, M,
        show_eps=True,
        title=rf"$G_{{\theta^{{{t}}}}}^\varepsilon$  —  {cnt}/{tot} aristas con $d_{{circ}}\leq {eps}$"
    )
    if eps_synced:
        ax_eps.set_facecolor(C_SYNC_BG)

    draw_metrics(ax_met, r_hist, a_hist, t, sync_t)

    if exact_synced:
        ax_met.set_title("θ ∈ Δ  ✓", fontsize=10, color=C_OK)
    elif eps_synced:
        ax_met.set_title(r"θ ∈ Δ$^\varepsilon$  ✓", fontsize=10, color=C_OK)
    else:
        ax_met.set_title(r"θ ∉ Δ$^\varepsilon$  ✗", fontsize=10, color=C_BAD)

    draw_subgraph_row(fig, gs[1, :], G, pos, history, M, eps, stimes, t)

    sync_info = f"ε-sync en t = {sync_t}" if sync_t is not None else "sin ε-sync"
    fig.suptitle(
        r"$G_{\theta^0}^{\varepsilon} \to \cdots \to G_{\theta^T}^{\varepsilon}$"
        + f"     [ε={eps}; {sync_info}]",
        fontsize=12, y=0.995
    )

    plt.show()


# ============================================================
# 9. INTERFAZ IPYWIDGETS
# ============================================================
_ST = {"description_width": "108px"}

w_N = widgets.IntSlider(
    value=12, min=3, max=50, step=1,
    description="N (nodos)", style=_ST
)

w_M = widgets.IntSlider(
    value=6, min=2, max=12, step=1,
    description="M (fases)", style=_ST
)

w_eps = widgets.IntSlider(
    value=0, min=0, max=10.0, step=0.05,
    description="ε", style=_ST
)

w_K = widgets.FloatSlider(
    value=1.2, min=0.0, max=4.0, step=0.05,
    description="K (acoplam.)",
    readout_format=".2f", style=_ST
)

w_om = widgets.IntSlider(
    value=1, min=0, max=5, step=1,
    description="ω (frecuencia)", style=_ST
)

w_T = widgets.IntSlider(
    value=30, min=5, max=1000, step=1,
    description="T (pasos)", style=_ST
)

w_seed = widgets.IntSlider(
    value=7, min=0, max=200, step=1,
    description="Semilla", style=_ST
)

w_t = widgets.IntSlider(
    value=0, min=0, max=1000, step=1,
    description="t visualizar", style=_ST
)

w_init = widgets.Dropdown(
    options=["bloques", "aleatoria", "gradiente", "casi_sincronizada"],
    value="bloques",
    description="Cond. inicial",
    style=_ST,
    layout=widgets.Layout(width="260px")
)

w_topo = widgets.ToggleButtons(
    options=["camino", "ciclo", "completo", "estrella", "aleatorio", "grid", "hex"],
    value="grid",
    description="Topología",
    button_style="",
    style={"description_width": "80px", "button_width": "84px"}
)

btn_sim = widgets.Button(
    description="▶  Simular",
    button_style="success",
    layout=widgets.Layout(width="120px")
)

btn_step = widgets.Button(
    description="Paso +1",
    layout=widgets.Layout(width="100px")
)

btn_reset = widgets.Button(
    description="Reiniciar",
    button_style="warning",
    layout=widgets.Layout(width="110px")
)

btn_rand = widgets.Button(
    description="Cond. aleatoria",
    layout=widgets.Layout(width="145px")
)


btn_case_sync = widgets.Button(
    description="Caso sugerido sync",
    button_style="info",
    layout=widgets.Layout(width="150px")
)


html_info = widgets.HTML(value="")
out = widgets.Output()

_state = dict(
    history=None,
    G=None,
    pos=None,
    M=6,
    eps=0,
    r_hist=[],
    a_hist=[],
    sync_t=None,
    exact_t=None
)


def _set_info(msg: str, color: str = "black"):
    html_info.value = f'<span style="font-size:12px;color:{color}">{msg}</span>'


# ============================================================
# 10. LÓGICA PRINCIPAL
# ============================================================
def _run(rand_init: bool = False):
    """
    Construye grafo, simula y actualiza _state.
    No toca w_t, salvo para ajustar límites.
    """
    N, M = w_N.value, w_M.value
    eps = w_eps.value
    K, om = w_K.value, w_om.value
    T = w_T.value
    topo = w_topo.value
    seed = w_seed.value
    init = "aleatoria" if rand_init else w_init.value

    G = build_graph(topo, N, seed=seed)
    theta0 = make_theta0(N, M, topo, mode=init, seed=seed)
    pos = build_positions(topo, N)
    history = simulate(G, theta0, M, K, om, T)

    r_hist = [order_parameter(th, M) for th in history]

    edge_stats = [edge_agreement(G, th, M, eps) for th in history]
    a_hist = [cnt / max(tot, 1) for cnt, tot in edge_stats]

    sync_t = first_sync_time(history, M, eps, exact=False)
    exact_t = first_sync_time(history, M, 0, exact=True)

    _state.update(
        history=history,
        G=G,
        pos=pos,
        M=M,
        eps=eps,
        r_hist=r_hist,
        a_hist=a_hist,
        sync_t=sync_t,
        exact_t=exact_t
    )

    w_t.max = T
    if w_t.value > T:
        w_t.value = 0

    if sync_t is not None:
        msg = f"✓ ε-sincronización alcanzada en t = {sync_t} / {T}"
        if exact_t is not None:
            msg += f" · sync exacta en t = {exact_t}"
        _set_info(msg, C_OK)
    else:
        note = ""
        if om % M != 0:
            note = f" (ω={om} no es múltiplo de M={M}; esto puede impedir sync exacta)"
        _set_info(f"✗ Sin ε-sync en T={T} pasos.{note}", C_WARN)

    return history, G, pos, M, eps, r_hist, a_hist, sync_t


def _render():
    s = _state
    if s["history"] is None:
        return

    make_figure(
        s["G"],
        s["history"],
        w_t.value,
        s["M"],
        s["eps"],
        s["pos"],
        s["r_hist"],
        s["a_hist"],
        s["sync_t"]
    )


# ============================================================
# 11. CALLBACKS
# ============================================================
def on_simulate(_=None):
    with out:
        clear_output(wait=True)
        history, G, pos, M, eps, r_hist, a_hist, sync_t = _run(rand_init=False)
        _render()
        print(f"θ⁰ = {history[0].tolist()}")
        print(f"θᵀ = {history[-1].tolist()}")

        exact_t = _state.get("exact_t", None)
        if exact_t is not None:
            print(f"Sincronización exacta en t={exact_t}.")
        elif sync_t is not None:
            print(f"ε-sincronización en t={sync_t} con ε={eps}.")

        print(f"Estado final de la red: {'ε-sincronizada' if sync_t is not None else 'no sincronizada'}")

def on_step(_=None):
    if _state["history"] is None:
        _set_info("Primero ejecuta la simulación.", "#888")
        return

    if w_t.value >= len(_state["history"]) - 1:
        _set_info("Fin de la simulación. Aumenta T y vuelve a simular.", "#888")
        return

    w_t.value += 1


def on_reset(_=None):
    if _state["history"] is None:
        on_simulate()
        return
    w_t.value = 0


def on_rand(_=None):
    with out:
        clear_output(wait=True)
        _run(rand_init=True)
        if w_t.value != 0:
            w_t.value = 0
        else:
            _render()


def on_update_t(change=None):
    if _state["history"] is None:
        return
    with out:
        clear_output(wait=True)
        _render()


def on_topo_change(change=None):
    with out:
        clear_output(wait=True)
        _run(rand_init=False)
        if w_t.value != 0:
            w_t.value = 0
        else:
            _render()


def on_hot_change(change=None):
    """
    K, ω y ε re-simulan con la misma estructura base.
    """
    if _state["history"] is None:
        return
    with out:
        clear_output(wait=True)
        _run(rand_init=False)
        _render()


def on_M_change(change=None):
    w_eps.max = max(0, w_M.value // 2)
    if w_eps.value > w_eps.max:
        w_eps.value = w_eps.max


def on_case_sync(_=None):
    """
    Carga un caso sugerido donde la red suele ε-sincronizarse
    en una topología tipo grid.
    """
    w_topo.value = "grid"
    w_N.value = 9
    w_M.value = 5
    w_eps.value = 1.0
    w_K.value = 0.80
    w_om.value = 0
    w_T.value = 20
    w_seed.value = 3
    w_init.value = "casi_sincronizada"

    with out:
        clear_output(wait=True)
        _run(rand_init=False)
        w_t.value = 0
        _render()

        history = _state["history"]
        sync_t = _state["sync_t"]
        exact_t = _state["exact_t"]

        print("Caso sugerido cargado:")
        print(f"Topología = {w_topo.value}")
        print(f"N = {w_N.value}, M = {w_M.value}, ε = {w_eps.value}")
        print(f"K = {w_K.value:.2f}, ω = {w_om.value}, T = {w_T.value}")
        print(f"θ⁰ = {history[0].tolist()}")
        print(f"θᵀ = {history[-1].tolist()}")

        if exact_t is not None:
            print(f"Sincronización exacta en t={exact_t}.")
        elif sync_t is not None:
            print(f"ε-sincronización en t={sync_t}.")
        else:
            print("Este caso no sincronizó en el horizonte elegido.")


# ============================================================
# 12. OBSERVERS
# ============================================================
btn_sim.on_click(on_simulate)
btn_step.on_click(on_step)
btn_reset.on_click(on_reset)
btn_rand.on_click(on_rand)
btn_case_sync.on_click(on_case_sync)

w_t.observe(on_update_t, names="value")
w_topo.observe(on_topo_change, names="value")
w_K.observe(on_hot_change, names="value")
w_om.observe(on_hot_change, names="value")
w_eps.observe(on_hot_change, names="value")
w_M.observe(on_M_change, names="value")


# ============================================================
# 13. LAYOUT
# ============================================================
_header = widgets.HTML("""
<div style="margin:4px 0 10px">
  <b style="font-size:15px">Luciérnagas — red discreta de Kuramoto</b><br>
  <span style="font-size:11px;color:#666">
    θᵢᵗ⁺¹ = (θᵢᵗ + ω + ⌊K Σ<sub>j∈N(i)</sub> δ<sub>M</sub>(θⱼ,θᵢ)⌋) mod M
    &nbsp;·&nbsp;
    G<sub>θ</sub><sup>ε</sup> = subgrafo de aristas con d<sub>circ</sub>(θᵤ,θᵥ) ≤ ε
  </span><br>
  <span style="font-size:11px;color:#888">
    K, ω y ε re-simulan con los valores nuevos.
    N, M, T, topología, semilla y condición inicial requieren nueva simulación.
  </span>
</div>
""")

_legend = widgets.HTML("""
<div style="font-size:11px; color:#444; margin:4px 0 8px">
<b>Interpretación de colores:</b><br>
<span style="color:#FFD600">●</span> θ = 0 → destello (fase de disparo)<br>
<span style="color:#888">●</span> Otros colores → fases en Z<sub>M</sub> (mapa circular)<br>
Mismo color = misma fase · Cambio de color = evolución temporal
</div>
""")

_row0 = widgets.HBox([w_N, w_M, w_eps, w_K, w_om])
_row1 = widgets.HBox([w_T, w_seed, w_init])
_row2 = widgets.HBox([w_t])
_row3 = widgets.HBox(
    [btn_sim, btn_step, btn_reset, btn_rand, btn_case_sync],
    layout=widgets.Layout(gap="8px")
)

ui = widgets.VBox([
    _header,
    _legend,
    w_topo,
    _row0,
    _row1,
    _row2,
    _row3,
    html_info,
    out
])

display(ui)

# Ajuste inicial de ε según M
on_M_change()

# Ejecutar al cargar
on_simulate()